# Docker — Local Build & Smoke Testing

This notebook builds the Docker image locally and runs smoke tests against the containerised API.

**Prerequisites:**
- Dev Container is running with Docker-outside-of-Docker
- `artifacts/model.pkl` exists (run `03_ml_pipeline.ipynb` → Section 1 first)
- Docker Desktop is running on the host machine

---

## Flow
```
Verify Docker → Build image → Run container → Smoke test → Clean up
```

In [ ]:
import os
import time
import json
import subprocess

ROOT = "/workspaces/marketing-model-mlops-azure"
IMAGE_TAG = "bank-marketing-api:local"
CONTAINER_NAME = "bm-smoke-test"
API_PORT = 8000

os.chdir(ROOT)
print(f"Working directory: {os.getcwd()}")
print(f"Image tag:         {IMAGE_TAG}")
print(f"Container name:    {CONTAINER_NAME}")

## 1. Verify Docker Access

Confirm the Docker CLI can reach the host daemon via Docker-outside-of-Docker.

In [ ]:
%%bash
echo "=== Docker version ==="
docker --version

echo ""
echo "=== Docker daemon info ==="
docker info 2>&1 | grep -E 'Server Version|Operating System|Total Memory' || echo "ERROR: Cannot connect to Docker daemon — is Docker Desktop running on the host?"

## 2. Verify Model Artifact Exists

The Dockerfile copies `artifacts/model.pkl` into the image. It must exist before building.

In [ ]:
model_path = os.path.join(ROOT, "artifacts/model.pkl")

if os.path.exists(model_path):
    size_kb = os.path.getsize(model_path) / 1024
    print(f"OK  artifacts/model.pkl found ({size_kb:.1f} KB)")
else:
    print("MISSING  artifacts/model.pkl")
    print("Run:  python main.py train")
    print("Or run 03_ml_pipeline.ipynb Section 1 first.")
    raise FileNotFoundError("artifacts/model.pkl not found — train the model before building the image.")

## 3. Build the Docker Image

Builds `bank-marketing-api:local` from the project `Dockerfile`.

To force a clean rebuild (e.g. after updating `requirements.txt`), add `--no-cache` to the command below.

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

echo "Building image: bank-marketing-api:local"
docker build -t bank-marketing-api:local .

echo ""
echo "=== Image created ==="
docker images | grep bank-marketing-api

## 4. Run the Container

Start the API container in the background and wait for it to finish loading the model.

> **DooD networking note:** In Docker-outside-of-Docker, `docker run -p 8000:8000` publishes the port to the *host machine's* localhost — not to `localhost` inside the devcontainer. All `curl` commands in this notebook run inside the devcontainer, so they would get "connection refused".
>
> Fix: `--network container:$(hostname)` shares the devcontainer's network namespace with the API container. Port 8000 then resolves correctly at `localhost:8000` from inside the devcontainer.

In [ ]:
%%bash
# Stop and remove any pre-existing container with the same name
docker rm -f bm-smoke-test 2>/dev/null || true

echo "Starting container..."
docker run -d \
  --name bm-smoke-test \
  --network container:$(hostname) \
  bank-marketing-api:local

# --network container:$(hostname) shares the devcontainer's network namespace.
# Without this, -p 8000:8000 publishes to the host machine's localhost (not
# reachable via curl from inside the devcontainer in DooD).

echo "Waiting 5 seconds for model to load..."
sleep 5

echo ""
echo "=== Container status ==="
docker ps --filter name=bm-smoke-test --format 'table {{.Names}}\t{{.Status}}'

## 5. Health Check

Confirm the API is alive and the model loaded correctly.

In [ ]:
%%bash
echo "=== Health check ==="
curl -sf http://localhost:8000/health | python3 -m json.tool

echo ""
echo "Expected: {\"status\": \"healthy\"}"

## 6. Prediction Smoke Test

Send a sample payload to `/predict` and verify the response structure.

In [ ]:
%%bash
echo "=== Prediction request ==="
curl -sf -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{
    "age": 35,
    "job": "management",
    "marital": "married",
    "education": "tertiary",
    "default": "no",
    "balance": 1500.0,
    "housing": "yes",
    "loan": "no",
    "contact": "cellular",
    "day": 15,
    "month": "may",
    "duration": 250.0,
    "campaign": 1,
    "pdays": -1,
    "previous": 0,
    "poutcome": "unknown"
  }' | python3 -m json.tool

echo ""
echo "Expected: {\"prediction\": ..., \"probability\": ..., \"label\": \"yes\" or \"no\"}"

## 7. Edge Case Tests

Validate API behaviour for boundary inputs and validation errors.

| Scenario | Expected |
|---|---|
| Young customer (age 18) | Valid prediction |
| Negative balance | Valid prediction (signed-log handles it) |
| Missing required field | HTTP 422 |
| Wrong field type | HTTP 422 |

In [ ]:
%%bash
BASE_URL="http://localhost:8000"

echo "--- Young customer (age 18) ---"
curl -sf -X POST $BASE_URL/predict \
  -H "Content-Type: application/json" \
  -d '{"age":18,"job":"student","marital":"single","education":"secondary","default":"no","balance":100.0,"housing":"no","loan":"no","contact":"cellular","day":5,"month":"jan","duration":60.0,"campaign":1,"pdays":-1,"previous":0,"poutcome":"unknown"}' \
  | python3 -m json.tool

echo ""
echo "--- Negative balance ---"
curl -sf -X POST $BASE_URL/predict \
  -H "Content-Type: application/json" \
  -d '{"age":45,"job":"blue-collar","marital":"married","education":"primary","default":"no","balance":-500.0,"housing":"yes","loan":"yes","contact":"telephone","day":10,"month":"jun","duration":120.0,"campaign":3,"pdays":-1,"previous":0,"poutcome":"unknown"}' \
  | python3 -m json.tool

echo ""
echo "--- Missing required field (age omitted) — expect HTTP 422 ---"
curl -s -o /dev/null -w "%{http_code}" -X POST $BASE_URL/predict \
  -H "Content-Type: application/json" \
  -d '{"job":"management"}'
echo " (expected: 422)"

echo ""
echo "--- Wrong field type (age as string) — expect HTTP 422 ---"
curl -s -o /dev/null -w "%{http_code}" -X POST $BASE_URL/predict \
  -H "Content-Type: application/json" \
  -d '{"age":"thirty","job":"management","marital":"married","education":"tertiary","default":"no","balance":1500.0,"housing":"yes","loan":"no","contact":"cellular","day":15,"month":"may","duration":250.0,"campaign":1,"pdays":-1,"previous":0,"poutcome":"unknown"}'
echo " (expected: 422)"

## 8. Scripted Pass/Fail Smoke Test

Replicates the CI smoke test logic: runs health check and prediction, exits non-zero on failure.

In [ ]:
%%bash
set -e

echo "=== Smoke test ==="

HEALTH=$(curl -sf http://localhost:8000/health)
echo "$HEALTH" | grep -q '"healthy"' || { echo "FAIL: health check"; exit 1; }
echo "PASS: health check"

PRED=$(curl -sf -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"age":35,"job":"management","marital":"married","education":"tertiary","default":"no","balance":1500.0,"housing":"yes","loan":"no","contact":"cellular","day":15,"month":"may","duration":250.0,"campaign":1,"pdays":-1,"previous":0,"poutcome":"unknown"}')
echo "$PRED" | grep -q '"prediction"' || { echo "FAIL: predict endpoint"; exit 1; }
echo "PASS: predict endpoint"

echo ""
echo "All smoke tests passed."
echo "Response: $PRED"

## 9. View Container Logs

Inspect API startup logs and any request logs.

In [ ]:
%%bash
docker logs bm-smoke-test

## 10. Clean Up

Stop and remove the smoke test container. Run this cell when done.

In [ ]:
%%bash
docker stop bm-smoke-test && docker rm bm-smoke-test
echo "Container stopped and removed."

---

## Summary

| Step | Command | Expected result |
|---|---|---|
| Verify Docker | `docker info` | Daemon reachable |
| Build image | `docker build -t bank-marketing-api:local .` | Image created |
| Run container | `docker run -d --name bm-smoke-test -p 8000:8000 ...` | Container running |
| Health check | `curl http://localhost:8000/health` | `{"status":"healthy"}` |
| Predict | `curl -X POST http://localhost:8000/predict ...` | `{"prediction":...}` |
| Edge cases | Missing/wrong fields | HTTP 422 |
| Clean up | `docker stop && docker rm` | Container removed |

Once the Docker smoke test passes, open **`05_kubernetes_setup.ipynb`** to deploy to the local kind cluster.